# Dual Swin V2 — Hybrid SegFormer×UNet++ Decoder (v6-DA: Domain-Adapted)

**Key change in v6-DA: Per-Image Percentile Normalization** to handle domain shift between training data (archive/) and test data (phase2_test/):
- **DEM catastrophic shift**: Train DEM range [-1345, 3110] vs Test DEM range [4275, 7232] — **zero overlap**
- **B4/B5/B6 moderate shift**: +20-26 DN units offset
- **Solution**: Each image's bands are normalized to their OWN P1/P99 percentiles → [0,1], making the model invariant to absolute value differences

**Architecture (same as v6):**
- Hybrid SegFormer×UNet++ Decoder with SE-gated adaptive fusion
- Deep supervision + Boundary-aware loss + Lovász
- Stronger augmentations + Gradient clipping + TTA

In [ ]:
# Install dependencies
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "timm", "albumentations", "rasterio", "opencv-python-headless",
                       "tqdm", "scikit-learn", "matplotlib"])

In [ ]:

# ============================================================
# Dual-Swin V2 (RGB + Aux4) — HYBRID SegFormer×UNet++ Decoder
# K-Fold Cross-Validation — ARCHIVE .tif DATASET VERSION  (v6-DA)
# ★ Domain-Adapted: Per-Image Percentile Normalization
# ============================================================

import os, json, time, zipfile, random, math, copy
from pathlib import Path

import numpy as np
import cv2
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import albumentations as A
import rasterio
import timm
from sklearn.model_selection import KFold

# ============================================================
# Device & Reproducibility
# ============================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

# ============================================================
# Channel mapping (archive dataset: CTX, Slope, DEM, B3, B4, B5, B6)
# ============================================================
CHANNEL_MAP = {
    "CTX": [0], "SLOPE": [1], "DEM": [2],
    "B3": [3], "B4": [4], "B5": [5], "B6": [6],
}
RGB_INDICES = [4, 5, 6]
AUX_INDICES = [0, 1, 2, 3]
ALL_INDICES = list(range(7))

EXP = {"id": "DualSwinV2_HybridSegFormerUNetPP_KFold_Archive_v6_DA",
       "channels": "all 7 archive/standardized — per-image normalization"}

# ============================================================
# Augmentations — STRONGER than v5
#   Added: ElasticTransform, GridDistortion, CoarseDropout
# ============================================================
def build_transforms(img_size, strong=True):
    if strong:
        geo_aug = A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Affine(
                translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
                scale=(0.85, 1.15),
                rotate=(-25, 25),
                p=0.5,
                border_mode=cv2.BORDER_REFLECT_101,
            ),
            A.ElasticTransform(alpha=30, sigma=5, p=0.2),
            A.GridDistortion(num_steps=5, distort_limit=0.15, p=0.2),
            A.CoarseDropout(
                num_holes_range=(2, 6),
                hole_height_range=(int(img_size * 0.04), int(img_size * 0.08)),
                hole_width_range=(int(img_size * 0.04), int(img_size * 0.08)),
                fill=0, p=0.2,
            ),
        ])
    else:
        geo_aug = A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Affine(
                translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
                scale=(0.90, 1.10),
                rotate=(-20, 20),
                border_mode=cv2.BORDER_REFLECT_101,
                mode=cv2.BORDER_REFLECT_101,
            ),
        ])
    rgb_photo = A.Compose([
        A.GaussianBlur(p=0.15),
        A.RandomBrightnessContrast(p=0.3),
    ])
    val_aug = A.Compose([A.Resize(img_size, img_size)])
    return geo_aug, rgb_photo, val_aug

# ============================================================
# Normalization stats — v3-style robust percentile + z-score
# ============================================================
def compute_scaling_stats(img_paths, band_indices, p_low=1.0, p_high=99.0, max_files=None):
    """Step 1: Per-band robust percentile clipping bounds (median across images)."""
    lows, highs = [], []
    paths = list(img_paths)
    if max_files is not None:
        paths = paths[:max_files]
    for p in tqdm(paths, desc="Compute band pctl stats"):
        with rasterio.open(str(p)) as src:
            arr = src.read(band_indices).astype(np.float32)
        flat = arr.reshape(arr.shape[0], -1)
        lo = np.percentile(flat, p_low, axis=1)
        hi = np.percentile(flat, p_high, axis=1)
        lows.append(lo); highs.append(hi)
    low  = np.median(np.stack(lows,  axis=0), axis=0)
    high = np.median(np.stack(highs, axis=0), axis=0)
    high = np.maximum(high, low + 1e-6)
    return {"low": low, "high": high, "p_low": p_low, "p_high": p_high}

def normalize_bands(arr_chw, low, high):
    """Step 2: Clip to [low, high] and rescale to [0, 1]."""
    x = arr_chw.astype(np.float32)
    low  = low[:, None, None]
    high = high[:, None, None]
    x = (x - low) / (high - low)
    return np.clip(x, 0.0, 1.0)

# ============================================================
# ★ v6-DA: Per-image percentile normalization (domain-invariant)
# ============================================================
def normalize_bands_per_image(arr_chw, p_low=1.0, p_high=99.0):
    """Per-image, per-band percentile normalization — domain-invariant.

    Each band is clipped to its OWN image's [P_low, P_high] percentiles and
    rescaled to [0, 1].  This eliminates sensitivity to absolute value shifts
    between domains (e.g. DEM at different elevations).
    """
    C = arr_chw.shape[0]
    out = np.empty_like(arr_chw, dtype=np.float32)
    for c in range(C):
        flat = arr_chw[c].ravel()
        lo = np.percentile(flat, p_low)
        hi = np.percentile(flat, p_high)
        hi = max(hi, lo + 1e-6)
        out[c] = np.clip((arr_chw[c].astype(np.float32) - lo) / (hi - lo), 0.0, 1.0)
    return out

def compute_mean_std_per_image_norm(img_paths, band_indices,
                                     p_low=1.0, p_high=99.0,
                                     max_files=None, max_pixels=2_000_000):
    """Compute channel mean/std AFTER per-image percentile normalization.

    These statistics are used for z-score standardization during training
    and must be saved / hardcoded for inference.
    """
    paths = list(img_paths)
    if max_files is not None:
        paths = paths[:max_files]
    C = len(band_indices)
    sums = np.zeros(C, dtype=np.float64)
    sqs  = np.zeros(C, dtype=np.float64)
    n    = 0
    rng  = np.random.default_rng(123)
    for p in tqdm(paths, desc="Compute mean/std (per-image norm)"):
        with rasterio.open(str(p)) as src:
            arr = src.read(band_indices).astype(np.float32)
        arr = normalize_bands_per_image(arr, p_low, p_high)
        flat = arr.reshape(C, -1)
        if flat.shape[1] > 20000:
            idx = rng.choice(flat.shape[1], size=20000, replace=False)
            flat = flat[:, idx]
        sums += flat.sum(axis=1)
        sqs  += (flat * flat).sum(axis=1)
        n    += flat.shape[1]
        if n >= max_pixels:
            break
    means = (sums / max(n, 1)).astype(np.float32)
    vars_ = (sqs  / max(n, 1) - means.astype(np.float64) ** 2).clip(min=1e-8)
    stds  = np.sqrt(vars_).astype(np.float32)
    print(f"Channel means (per-image norm): {means}")
    print(f"Channel stds  (per-image norm): {stds}")
    return means, stds

# (Legacy — kept for reference but no longer used in v6-DA)
def compute_mean_std_after_scaling(img_paths, band_indices, low, high,
                                   max_files=None, max_pixels=2_000_000):
    """Step 3: Compute mean/std AFTER [0,1] rescaling for z-score standardization."""
    paths = list(img_paths)
    if max_files is not None:
        paths = paths[:max_files]
    C = len(band_indices)
    sums = np.zeros(C, dtype=np.float64)
    sqs  = np.zeros(C, dtype=np.float64)
    n    = 0
    rng  = np.random.default_rng(123)
    for p in tqdm(paths, desc="Compute mean/std after scaling"):
        with rasterio.open(str(p)) as src:
            arr = src.read(band_indices).astype(np.float32)
        arr  = normalize_bands(arr, low, high)
        flat = arr.reshape(C, -1)
        if flat.shape[1] > 20000:
            idx = rng.choice(flat.shape[1], size=20000, replace=False)
            flat = flat[:, idx]
        sums += flat.sum(axis=1)
        sqs  += (flat * flat).sum(axis=1)
        n    += flat.shape[1]
        if n >= max_pixels:
            break
    means = (sums / max(n, 1)).astype(np.float32)
    vars_ = (sqs  / max(n, 1) - means.astype(np.float64) ** 2).clip(min=1e-8)
    stds  = np.sqrt(vars_).astype(np.float32)
    print(f"Channel means (after scaling): {means}")
    print(f"Channel stds  (after scaling): {stds}")
    return means, stds

# ============================================================
# Dataset — reads .tif files via rasterio, applies v3-style normalization
# ============================================================
class MarsDualModalSegDataset(Dataset):
    def __init__(self, img_paths, mask_paths, rgb_indices, aux_indices,
                 scaling_stats, mean_all, std_all,
                 geo_aug=None, rgb_photo_aug=None, val_aug=None, is_train=True):
        self.img_paths = list(img_paths)
        self.mask_paths = list(mask_paths) if mask_paths is not None else None
        self.rgb_indices = rgb_indices
        self.aux_indices = aux_indices
        # v6-DA: global percentile bounds no longer used — per-image normalization instead
        # self.low  = scaling_stats["low"]
        # self.high = scaling_stats["high"]
        self.mean_all = mean_all
        self.std_all = std_all
        self.geo_aug = geo_aug
        self.rgb_photo_aug = rgb_photo_aug
        self.val_aug = val_aug
        self.is_train = is_train

    def __len__(self):
        return len(self.img_paths)

    def _standardize(self, x_chw, indices):
        mean = self.mean_all[indices][:, None, None]
        std  = self.std_all[indices][:, None, None]
        return (x_chw - mean) / std

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        with rasterio.open(str(img_path)) as src:
            arr = src.read().astype(np.float32)  # (C, H, W)
        # ★ v6-DA: per-image percentile normalization (domain-invariant)
        arr = normalize_bands_per_image(arr)
        arr_hwc = np.transpose(arr, (1, 2, 0))

        mask = None
        if self.mask_paths is not None:
            with rasterio.open(str(self.mask_paths[idx])) as src:
                mask = src.read(1).astype(np.float32)  # (H, W)

        if self.is_train:
            if self.geo_aug is not None:
                aug = self.geo_aug(image=arr_hwc, mask=mask)
                arr_hwc = aug["image"]; mask = aug["mask"]
            if self.rgb_photo_aug is not None:
                rgb_ch = arr_hwc[..., self.rgb_indices]
                aux_ch = arr_hwc[..., self.aux_indices]
                rgb_ch = self.rgb_photo_aug(image=rgb_ch)["image"]
                out = np.empty_like(arr_hwc)
                for i, ci in enumerate(self.rgb_indices):
                    out[..., ci] = rgb_ch[..., i]
                for i, ci in enumerate(self.aux_indices):
                    out[..., ci] = aux_ch[..., i]
                arr_hwc = out
        else:
            if self.val_aug is not None:
                if mask is not None:
                    aug = self.val_aug(image=arr_hwc, mask=mask)
                    arr_hwc = aug["image"]; mask = aug["mask"]
                else:
                    arr_hwc = self.val_aug(image=arr_hwc)["image"]

        arr_chw = np.transpose(arr_hwc, (2, 0, 1))
        rgb = self._standardize(arr_chw[self.rgb_indices], self.rgb_indices)
        aux = self._standardize(arr_chw[self.aux_indices], self.aux_indices)
        rgb_t = torch.from_numpy(rgb).float()
        aux_t = torch.from_numpy(aux).float()
        if mask is not None:
            mask_t = torch.from_numpy(mask).float().unsqueeze(0)
            return rgb_t, aux_t, mask_t
        else:
            return rgb_t, aux_t, str(img_path)

# ============================================================
# Pos_weight — reads .tif masks via rasterio
# ============================================================
def compute_pos_weight(mask_paths):
    fg = 0; tot = 0
    for p in tqdm(mask_paths, desc="Compute pos_weight"):
        with rasterio.open(str(p)) as src:
            m = src.read(1)
        m01 = (m > 0).astype(np.uint8)
        fg += int(m01.sum()); tot += int(m01.size)
    frac = fg / tot
    pos_weight = (1.0 - frac) / (frac + 1e-9)
    return float(frac), float(pos_weight)
# ============================================================
# Metrics
# ============================================================
@torch.no_grad()
def compute_leaderboard_metrics_from_loader(model, loader, thresh=0.5):
    model.eval()
    TP = FP = FN = TN = 0.0
    for rgb, aux, mask in tqdm(loader, desc="ValMetric", leave=False):
        rgb  = rgb.to(DEVICE, non_blocking=True)
        aux  = aux.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)
        out = model(rgb, aux)
        logits = out["logits"] if isinstance(out, dict) else out
        pred = (torch.sigmoid(logits) > thresh).float()
        TP += (pred * mask).sum().item()
        FP += (pred * (1 - mask)).sum().item()
        FN += ((1 - pred) * mask).sum().item()
        TN += ((1 - pred) * (1 - mask)).sum().item()
    eps = 1e-7
    iou_fg = TP / (TP + FP + FN + eps)
    iou_bg = TN / (TN + FP + FN + eps)
    miou   = 0.5 * (iou_fg + iou_bg)
    prec_fg = TP / (TP + FP + eps)
    rec_fg  = TP / (TP + FN + eps)
    f1_fg   = 2 * prec_fg * rec_fg / (prec_fg + rec_fg + eps)
    return {
        "IoU_fg": float(iou_fg), "IoU_bg": float(iou_bg), "mIoU": float(miou),
        "Precision_fg": float(prec_fg), "Recall_fg": float(rec_fg), "F1_fg": float(f1_fg),
        "TP": float(TP), "FP": float(FP), "FN": float(FN), "TN": float(TN),
    }

# ============================================================
# Loss: BCE(pos_weight) + Dice + Boundary
# ============================================================
def dice_loss(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    num = 2 * (probs * targets).sum(dim=(2, 3))
    den = (probs + targets).sum(dim=(2, 3)) + eps
    return 1 - (num / den).mean()

def boundary_loss(logits, targets, kernel_size=3):
    """Boundary-aware loss: upweight pixels near mask boundaries."""
    probs = torch.sigmoid(logits)
    # Extract boundaries via morphological gradient
    pad = kernel_size // 2
    kernel = torch.ones(1, 1, kernel_size, kernel_size, device=targets.device)
    dilated = F.conv2d(targets, kernel, padding=pad).clamp(0, 1)
    eroded  = 1.0 - F.conv2d(1.0 - targets, kernel, padding=pad).clamp(0, 1)
    boundary_map = (dilated - eroded).clamp(0, 1)  # 1 at boundary, 0 elsewhere
    # Weight: 3× at boundaries, 1× elsewhere
    weight = 1.0 + 2.0 * boundary_map
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    return (bce * weight).mean()

def lovasz_grad(gt_sorted):
    """Compute Lovász gradient for sorted ground-truth."""
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1.0 - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard

def lovasz_hinge_flat(logits, labels):
    """Binary Lovász hinge loss — directly optimizes IoU."""
    if len(labels) == 0:
        return logits.sum() * 0.0
    signs = 2.0 * labels.float() - 1.0
    errors = 1.0 - logits * signs
    errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
    perm = perm.data
    gt_sorted = labels[perm]
    grad = lovasz_grad(gt_sorted)
    loss = torch.dot(F.relu(errors_sorted), grad)
    return loss

def lovasz_hinge(logits, labels):
    """Per-image Lovász hinge, then average."""
    losses = []
    for log, lab in zip(logits.squeeze(1), labels.squeeze(1)):
        losses.append(lovasz_hinge_flat(log.reshape(-1), lab.reshape(-1)))
    return torch.stack(losses).mean()

class HybridBCEDiceBoundaryLoss(nn.Module):
    """
    Combined loss: α·BCE + β·Dice + γ·Boundary + δ·Lovász
    Boundary loss upweights edges; Lovász directly optimizes IoU.
    """
    def __init__(self, pos_weight: float,
                 alpha=0.3, beta=0.3, gamma=0.2, delta=0.2):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([pos_weight], device=DEVICE))
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.delta = delta

    def forward(self, logits, targets):
        l_bce  = self.bce(logits, targets)
        l_dice = dice_loss(logits, targets)
        l_bnd  = boundary_loss(logits, targets)
        l_lov  = lovasz_hinge(logits, targets)
        return (self.alpha * l_bce + self.beta * l_dice +
                self.gamma * l_bnd + self.delta * l_lov)

# ============================================================
# Deep Supervision loss wrapper
# ============================================================
class DeepSupervisionLoss(nn.Module):
    """
    Wraps a base loss and adds weighted auxiliary losses from
    intermediate decoder outputs.
    """
    def __init__(self, base_loss_fn, aux_weight=0.3):
        super().__init__()
        self.base_loss = base_loss_fn
        self.aux_weight = aux_weight

    def forward(self, model_output, targets):
        if isinstance(model_output, dict):
            logits = model_output["logits"]
            main_loss = self.base_loss(logits, targets)
            aux_logits_list = model_output.get("aux_logits", [])
            if len(aux_logits_list) > 0:
                aux_loss = 0.0
                for aux_logits in aux_logits_list:
                    # Resize aux to match target size if needed
                    if aux_logits.shape[-2:] != targets.shape[-2:]:
                        aux_logits = F.interpolate(
                            aux_logits, size=targets.shape[-2:],
                            mode="bilinear", align_corners=False)
                    aux_loss += self.base_loss(aux_logits, targets)
                aux_loss /= len(aux_logits_list)
                return main_loss + self.aux_weight * aux_loss
            return main_loss
        else:
            return self.base_loss(model_output, targets)

# ============================================================
# EMA
# ============================================================
class EMA:
    def __init__(self, model: nn.Module, decay=0.995, warmup_steps=0):
        self.decay = decay
        self.warmup_steps = warmup_steps
        self.step_count = 0
        self.shadow = {}
        self.backup = {}
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[name] = p.data.clone()

    def _get_decay(self):
        if self.warmup_steps > 0 and self.step_count < self.warmup_steps:
            return min(self.decay, 1.0 - 1.0 / (self.step_count + 1))
        return self.decay

    @torch.no_grad()
    def update(self, model: nn.Module):
        d = self._get_decay()
        self.step_count += 1
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            assert name in self.shadow
            new_avg = (1.0 - d) * p.data + d * self.shadow[name]
            self.shadow[name] = new_avg.clone()

    def apply_shadow(self, model: nn.Module):
        self.backup = {}
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            self.backup[name] = p.data.clone()
            p.data = self.shadow[name].clone()

    def restore(self, model: nn.Module):
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            p.data = self.backup[name].clone()
        self.backup = {}

# ============================================================
# Backbone helpers
# ============================================================
def adapt_patch_embed_in_chans(model, in_chans_new):
    pe = model.patch_embed
    old_conv = pe.proj
    old_w = old_conv.weight.data
    embed_dim, old_in, kh, kw = old_w.shape
    assert old_in == 3, f"Expected 3ch pretrained, got {old_in}"
    new_conv = nn.Conv2d(
        in_chans_new, embed_dim,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=(old_conv.bias is not None),
    )
    with torch.no_grad():
        new_w = torch.zeros(embed_dim, in_chans_new, kh, kw, device=old_w.device)
        new_w[:, :3, :, :] = old_w
        if in_chans_new > 3:
            rgb_mean = old_w.mean(dim=1, keepdim=True)
            new_w[:, 3:, :, :] = rgb_mean.repeat(1, in_chans_new - 3, 1, 1)
        new_conv.weight.copy_(new_w)
        if old_conv.bias is not None:
            new_conv.bias.copy_(old_conv.bias.data)
    pe.proj = new_conv
    return model

def make_swin_features(encoder_name, pretrained=True, img_size=256):
    enc = timm.create_model(
        encoder_name,
        pretrained=pretrained,
        features_only=True,
        out_indices=(0, 1, 2, 3),
        img_size=img_size,
    )
    if hasattr(enc, "patch_embed"):
        enc.patch_embed.img_size = None
        if hasattr(enc.patch_embed, "strict_img_size"):
            enc.patch_embed.strict_img_size = False
    return enc

def to_nchw(feats, in_chs):
    out = []
    for f, c in zip(feats, in_chs):
        if f.ndim == 4 and f.shape[-1] == c and f.shape[1] != c:
            f = f.permute(0, 3, 1, 2).contiguous()
        out.append(f)
    return out

# ============================================================
# Common building block
# ============================================================
class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, s, p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

# ============================================================
# SE (Squeeze-and-Excitation) block
# ============================================================
class SEBlock(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        mid = max(ch // r, 8)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(ch, mid, 1), nn.ReLU(inplace=True),
            nn.Conv2d(mid, ch, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.se(x)

# ============================================================
# Channel Attention Modules: SE, ECA, CBAM
# ============================================================

# --- SE (Squeeze-and-Excitation) ---
class SE(nn.Module):
    """Squeeze-and-Excitation channel attention.
    Learns a global channel-wise weight via GAP → FC → ReLU → FC → Sigmoid.
    """
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, channels, 1, bias=False),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.fc(x)

# --- ECA (Efficient Channel Attention) ---
class ECA(nn.Module):
    """Efficient Channel Attention — uses 1D conv instead of FC, nearly parameter-free.
    Adaptive kernel size based on channel count.
    """
    def __init__(self, channels, gamma=2, b=1):
        super().__init__()
        t = int(abs((math.log2(channels) + b) / gamma))
        k = t if t % 2 else t + 1  # ensure odd kernel
        k = max(k, 3)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=k // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        y = self.avg_pool(x)                         # (B, C, 1, 1)
        y = y.squeeze(-1).transpose(-1, -2)          # (B, 1, C)
        y = self.conv(y)                             # (B, 1, C)
        y = y.transpose(-1, -2).unsqueeze(-1)        # (B, C, 1, 1)
        y = self.sigmoid(y)
        return x * y

# --- CBAM (Convolutional Block Attention Module) ---
class CBAM(nn.Module):
    """CBAM = Channel Attention + Spatial Attention."""
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        # Channel attention
        mid = max(channels // reduction, 8)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, channels, 1, bias=False),
        )
        # Spatial attention
        self.spatial = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False),
            nn.Sigmoid(),
        )
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # Channel attention
        avg_out = self.mlp(F.adaptive_avg_pool2d(x, 1))
        max_out = self.mlp(F.adaptive_max_pool2d(x, 1))
        ca = self.sigmoid(avg_out + max_out)
        x = x * ca
        
        # Spatial attention
        avg_s = x.mean(dim=1, keepdim=True)
        max_s, _ = x.max(dim=1, keepdim=True)
        sa = self.spatial(torch.cat([avg_s, max_s], dim=1))
        return x * sa

# ============================================================
# Attention Strategy Modules (Options A, B, D, E)
# ============================================================

# --- Option A: Input-level Channel Attention ---
class InputChannelAttention(nn.Module):
    """Attention applied to raw input before encoder.
    attn_type: 'se', 'eca', 'cbam', or 'none'
    """
    def __init__(self, in_channels, attn_type='se', reduction=16):
        super().__init__()
        self.attn_type = attn_type.lower()
        if self.attn_type == 'se':
            self.attn = SE(in_channels, reduction)
        elif self.attn_type == 'eca':
            self.attn = ECA(in_channels)
        elif self.attn_type == 'cbam':
            self.attn = CBAM(in_channels, reduction)
        elif self.attn_type == 'none':
            self.attn = nn.Identity()
        else:
            raise ValueError(f"Unknown input attention type: {attn_type}")
    
    def forward(self, x):
        return self.attn(x)

# --- Option B: Post-Encoder Attention ---
class PostEncoderAttention(nn.Module):
    """Attention applied to encoder outputs (all 4 stages).
    attn_type: 'se', 'eca', 'cbam', or 'none'
    """
    def __init__(self, channel_list, attn_type='se', reduction=16):
        super().__init__()
        self.attn_type = attn_type.lower()
        if self.attn_type == 'none':
            self.attns = nn.ModuleList([nn.Identity() for _ in channel_list])
        elif self.attn_type == 'se':
            self.attns = nn.ModuleList([SE(c, reduction) for c in channel_list])
        elif self.attn_type == 'eca':
            self.attns = nn.ModuleList([ECA(c) for c in channel_list])
        elif self.attn_type == 'cbam':
            self.attns = nn.ModuleList([CBAM(c, reduction) for c in channel_list])
        else:
            raise ValueError(f"Unknown post-encoder attention type: {attn_type}")
    
    def forward(self, feats):
        return [self.attns[i](f) for i, f in enumerate(feats)]

# --- Option D: Decoder Output Attention ---
class DecoderOutputAttention(nn.Module):
    """Attention applied to final decoder output before head.
    attn_type: 'se', 'eca', 'cbam', or 'none'
    """
    def __init__(self, channels, attn_type='se', reduction=16):
        super().__init__()
        self.attn_type = attn_type.lower()
        if self.attn_type == 'se':
            self.attn = SE(channels, reduction)
        elif self.attn_type == 'eca':
            self.attn = ECA(channels)
        elif self.attn_type == 'cbam':
            self.attn = CBAM(channels, reduction)
        elif self.attn_type == 'none':
            self.attn = nn.Identity()
        else:
            raise ValueError(f"Unknown decoder output attention type: {attn_type}")
    
    def forward(self, x):
        return self.attn(x)

# --- Option E: Intra-Encoder Attention (Inside Encoder) ---
class SwinWithIntraAttention(nn.Module):
    """Wrapper that applies channel attention AFTER each stage of the Swin encoder.
    
    This implements 'Inside Encoder' attention by injecting attention modules
    between the Swin transformer stages. The attention is applied to each
    stage's output before it's used for feature extraction.
    """
    def __init__(self, base_encoder, attn_type='none', reduction=16):
        super().__init__()
        self.base_encoder = base_encoder
        self.attn_type = attn_type.lower()
        self.feature_info = base_encoder.feature_info
        
        self.chs = base_encoder.feature_info.channels()
        
        if self.attn_type == 'none':
            self.stage_attns = nn.ModuleList([nn.Identity() for _ in self.chs])
        elif self.attn_type == 'se':
            self.stage_attns = nn.ModuleList([SE(c, reduction) for c in self.chs])
        elif self.attn_type == 'eca':
            self.stage_attns = nn.ModuleList([ECA(c) for c in self.chs])
        elif self.attn_type == 'cbam':
            self.stage_attns = nn.ModuleList([CBAM(c, reduction) for c in self.chs])
        else:
            raise ValueError(f"Unknown intra-encoder attention type: {attn_type}")
        
        self._hook_handles = []
        self._register_hooks()
    
    def _get_stage_modules(self):
        stages = []
        if hasattr(self.base_encoder, 'stages'):
            for i in range(len(self.base_encoder.stages)):
                stages.append(self.base_encoder.stages[i])
        elif hasattr(self.base_encoder, 'layers'):
            for i in range(len(self.base_encoder.layers)):
                stages.append(self.base_encoder.layers[i])
        return stages
    
    def _register_hooks(self):
        if self.attn_type == 'none':
            return
        stages = self._get_stage_modules()
        for i, stage in enumerate(stages):
            handle = stage.register_forward_hook(self._make_hook(i))
            self._hook_handles.append(handle)
    
    def _make_hook(self, stage_idx):
        def hook(module, input, output):
            if output.ndim == 4:
                c = self.chs[stage_idx]
                if output.shape[-1] == c and output.shape[1] != c:
                    out_nchw = output.permute(0, 3, 1, 2).contiguous()
                    out_attn = self.stage_attns[stage_idx](out_nchw)
                    output = out_attn.permute(0, 2, 3, 1).contiguous()
                else:
                    output = self.stage_attns[stage_idx](output)
            return output
        return hook
    
    def remove_hooks(self):
        for handle in self._hook_handles:
            handle.remove()
        self._hook_handles = []
    
    def forward(self, x):
        return self.base_encoder(x)
    
    def __getattr__(self, name):
        if name in ['base_encoder', 'attn_type', 'chs', 'stage_attns', 
                    '_hook_handles', 'feature_info']:
            return super().__getattr__(name)
        return getattr(self.base_encoder, name)

def make_swin_with_intra_attention(encoder_name, pretrained=True, img_size=256, 
                                    intra_attn='none', reduction=16):
    """Create Swin encoder with intra-encoder attention."""
    base_enc = timm.create_model(
        encoder_name,
        pretrained=pretrained,
        features_only=True,
        out_indices=(0, 1, 2, 3),
        img_size=img_size,
    )
    if hasattr(base_enc, 'patch_embed'):
        base_enc.patch_embed.img_size = None
        if hasattr(base_enc.patch_embed, 'strict_img_size'):
            base_enc.patch_embed.strict_img_size = False
    
    if intra_attn.lower() == 'none':
        return base_enc
    else:
        return SwinWithIntraAttention(base_enc, attn_type=intra_attn, reduction=reduction)

# Attention type options
ATTENTION_TYPES = ["none", "se", "eca", "cbam"]

# ============================================================
# HYBRID DECODER: SegFormer × UNet++
# ============================================================
#
# Design rationale:
#
# SegFormer's MLP decoder is great at:
#   - Efficiently projecting all backbone scales to a unified dimension
#   - Capturing global multi-scale context via simple concat-and-fuse
#   - Being lightweight (just 1×1 convs)
#
# UNet++'s decoder is great at:
#   - Progressive feature refinement through dense nested skip connections
#   - Preserving fine spatial details and boundary information
#   - Propagating gradients via multiple short paths
#
# The hybrid approach:
#   1. SegFormer-style 1×1 MLP projections align all 4 scales → common dim C
#   2. UNet++ dense nested topology refines those unified features progressively
#   3. A parallel SegFormer global branch fuses all-scales-to-one for context
#   4. SE-gated attention merges UNet++ local detail + SegFormer global context
#   5. Deep supervision on intermediate UNet++ nodes improves boundaries
# ============================================================

class HybridSegFormerUNetPPDecoder(nn.Module):
    """
    Hybrid SegFormer × UNet++ Decoder

    Two parallel paths:
      • SegFormer path: MLP project all scales → concat → 1×1 fuse  (global context)
      • UNet++ path:    dense nested skips on projected features     (local refinement)

    Adaptive SE-gated fusion merges both paths.
    Returns main output + auxiliary logits for deep supervision.
    """

    def __init__(self, in_channels_list, fpn_channels=256):
        super().__init__()
        C = fpn_channels

        # ── SegFormer-style 1×1 MLP projections (shared by both branches) ──
        self.mlp_projs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(c, C, 1, bias=False),
                nn.BatchNorm2d(C),
                nn.ReLU(inplace=True),
            ) for c in in_channels_list
        ])

        # ── SegFormer global branch: concat all scales → 1×1 fuse ──
        self.seg_fuse = nn.Sequential(
            nn.Conv2d(C * 4, C, 1, bias=False),
            nn.BatchNorm2d(C),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1),
        )

        # ── UNet++ dense nested skip connection nodes ──
        # Topology (4 backbone stages → 0-indexed):
        #   Row 0: x00 ── x01 ── x02 ── x03    (highest resolution)
        #   Row 1: x10 ── x11 ── x12
        #   Row 2: x20 ── x21
        #   Row 3: x30
        #
        # x{i}0 = mlp_proj[i](feat[i])  (the backbone column)
        # Subsequent columns progressively merge neighbors.

        def _node(n_cat):
            """Conv block for a UNet++ node receiving n_cat concatenated inputs."""
            return nn.Sequential(
                ConvBNReLU(C * n_cat, C),
                ConvBNReLU(C, C),
            )

        # Column 1 nodes (each merges with 1 upsampled neighbor → 2 inputs)
        self.x01 = _node(2)
        self.x11 = _node(2)
        self.x21 = _node(2)

        # Column 2 nodes (merge original + col1 + upsampled col1-below → 3 inputs)
        self.x02 = _node(3)
        self.x12 = _node(3)

        # Column 3 node (merge x00 + x01 + x02 + up(x12) → 4 inputs)
        self.x03 = _node(4)

        # UNet++ branch final
        self.upp_final = nn.Sequential(ConvBNReLU(C, C), nn.Dropout2d(0.1))

        # ── Adaptive SE-gated fusion of SegFormer + UNet++ branches ──
        self.gate_fuse = nn.Sequential(
            nn.Conv2d(2 * C, C, 1, bias=False),
            nn.BatchNorm2d(C),
            nn.ReLU(inplace=True),
        )
        self.gate_se = SEBlock(C, r=16)

        # ── Deep supervision auxiliary heads (on intermediate UNet++ nodes) ──
        self.aux_head_01 = nn.Conv2d(C, 1, 1)
        self.aux_head_02 = nn.Conv2d(C, 1, 1)

    @staticmethod
    def _up(x, target):
        return F.interpolate(x, size=target.shape[-2:],
                             mode="bilinear", align_corners=False)

    def forward(self, feats):
        """
        feats: list of 4 tensors [c1, c2, c3, c4] from backbone, NCHW.
        Returns: (fused_features, aux_features_list)
        """
        # Step 1: SegFormer-style MLP projection → uniform channels
        x00 = self.mlp_projs[0](feats[0])   # highest-res
        x10 = self.mlp_projs[1](feats[1])
        x20 = self.mlp_projs[2](feats[2])
        x30 = self.mlp_projs[3](feats[3])   # lowest-res

        # ── SegFormer global branch ──
        target_size = x00.shape[-2:]
        seg_global = self.seg_fuse(torch.cat([
            x00,
            F.interpolate(x10, size=target_size, mode="bilinear", align_corners=False),
            F.interpolate(x20, size=target_size, mode="bilinear", align_corners=False),
            F.interpolate(x30, size=target_size, mode="bilinear", align_corners=False),
        ], dim=1))

        # ── UNet++ dense nested skip connections ──
        # Column 1
        x21 = self.x21(torch.cat([x20, self._up(x30, x20)], dim=1))
        x11 = self.x11(torch.cat([x10, self._up(x20, x10)], dim=1))
        x01 = self.x01(torch.cat([x00, self._up(x10, x00)], dim=1))

        # Column 2
        x12 = self.x12(torch.cat([x10, x11, self._up(x21, x10)], dim=1))
        x02 = self.x02(torch.cat([x00, x01, self._up(x11, x00)], dim=1))

        # Column 3 (final UNet++ output at highest resolution)
        x03 = self.x03(torch.cat([x00, x01, x02, self._up(x12, x00)], dim=1))
        upp_out = self.upp_final(x03)

        # ── SE-gated fusion: UNet++ local + SegFormer global ──
        fused = self.gate_fuse(torch.cat([upp_out, seg_global], dim=1))
        fused = self.gate_se(fused)

        # ── Auxiliary outputs for deep supervision ──
        aux_list = [self.aux_head_01(x01), self.aux_head_02(x02)]

        return fused, aux_list


# ============================================================
# Keep original decoders for comparison / ablation
# ============================================================
class SegFormerMLPDecoder(nn.Module):
    def __init__(self, in_channels_list, fpn_channels=256):
        super().__init__()
        self.linear_projs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(c, fpn_channels, 1, bias=False),
                nn.BatchNorm2d(fpn_channels),
                nn.ReLU(inplace=True),
            ) for c in in_channels_list
        ])
        self.fuse = nn.Sequential(
            nn.Conv2d(fpn_channels * 4, fpn_channels, 1, bias=False),
            nn.BatchNorm2d(fpn_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1),
        )
    def forward(self, feats):
        target_size = feats[0].shape[-2:]
        outs = []
        for i, f in enumerate(feats):
            x = self.linear_projs[i](f)
            if x.shape[-2:] != target_size:
                x = F.interpolate(x, size=target_size, mode="bilinear", align_corners=False)
            outs.append(x)
        return self.fuse(torch.cat(outs, dim=1))

class UNetPlusPlusDecoder(nn.Module):
    def __init__(self, in_channels_list, fpn_channels=256):
        super().__init__()
        C = fpn_channels
        self.reduce = nn.ModuleList([
            ConvBNReLU(c, C, k=1, s=1, p=0) for c in in_channels_list
        ])
        def _node(n_in):
            return nn.Sequential(ConvBNReLU(C * n_in, C), ConvBNReLU(C, C))
        self.x01 = _node(2); self.x11 = _node(2); self.x21 = _node(2)
        self.x02 = _node(3); self.x12 = _node(3)
        self.x03 = _node(4)
        self.final = nn.Sequential(ConvBNReLU(C, C), nn.Dropout2d(0.1))

    @staticmethod
    def _up(x, target):
        return F.interpolate(x, size=target.shape[-2:], mode="bilinear", align_corners=False)

    def forward(self, feats):
        x00, x10, x20, x30 = [self.reduce[i](f) for i, f in enumerate(feats)]
        x21 = self.x21(torch.cat([x20, self._up(x30, x20)], dim=1))
        x11 = self.x11(torch.cat([x10, self._up(x20, x10)], dim=1))
        x01 = self.x01(torch.cat([x00, self._up(x10, x00)], dim=1))
        x12 = self.x12(torch.cat([x10, x11, self._up(x21, x10)], dim=1))
        x02 = self.x02(torch.cat([x00, x01, self._up(x11, x00)], dim=1))
        x03 = self.x03(torch.cat([x00, x01, x02, self._up(x12, x00)], dim=1))
        return self.final(x03)

# ============================================================
# Decoder factory
# ============================================================
DECODER_LIST = ["hybrid_segformer_unetpp", "segformer_mlp", "unetplusplus"]

def build_decoder(name, in_channels_list, fpn_channels=256):
    name = name.lower()
    if name == "hybrid_segformer_unetpp":
        return HybridSegFormerUNetPPDecoder(in_channels_list, fpn_channels)
    if name == "segformer_mlp":
        return SegFormerMLPDecoder(in_channels_list, fpn_channels)
    if name == "unetplusplus":
        return UNetPlusPlusDecoder(in_channels_list, fpn_channels)
    raise ValueError(f"Unknown decoder: {name}")

# ============================================================
# Fusion strategies (kept from v5)
# ============================================================
class FusionBase(nn.Module):
    name = "base"
    def forward(self, feats_rgb, feats_aux):
        raise NotImplementedError

class FusionLateLogits(FusionBase):
    name = "late_logits"
    def __init__(self):
        super().__init__()
    def forward(self, feats_rgb, feats_aux):
        return feats_rgb, feats_aux

class FusionConcat1x1(FusionBase):
    name = "concat1x1"
    def __init__(self, chs):
        super().__init__()
        self.proj = nn.ModuleList([nn.Conv2d(2 * c, c, 1) for c in chs])
    def forward(self, A, B):
        return [self.proj[i](torch.cat([a, b], dim=1))
                for i, (a, b) in enumerate(zip(A, B))]

class FusionConcatSE(FusionBase):
    name = "concat_se"
    def __init__(self, chs, r=16):
        super().__init__()
        self.proj = nn.ModuleList()
        self.se   = nn.ModuleList()
        for c in chs:
            self.proj.append(nn.Sequential(
                nn.Conv2d(2 * c, c, 1, bias=False),
                nn.BatchNorm2d(c), nn.ReLU(inplace=True),
            ))
            mid = max(c // r, 8)
            self.se.append(nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Conv2d(c, mid, 1), nn.ReLU(inplace=True),
                nn.Conv2d(mid, c, 1), nn.Sigmoid(),
            ))
    def forward(self, A, B):
        out = []
        for i, (a, b) in enumerate(zip(A, B)):
            fused = self.proj[i](torch.cat([a, b], dim=1))
            w = self.se[i](fused)
            out.append(fused * w)
        return out

# --- Concat + ECA Fusion ---
class FusionConcatECA(FusionBase):
    name = "concat_eca"
    def __init__(self, chs):
        super().__init__()
        self.proj = nn.ModuleList()
        self.eca  = nn.ModuleList()
        for c in chs:
            self.proj.append(nn.Sequential(
                nn.Conv2d(2 * c, c, 1, bias=False),
                nn.BatchNorm2d(c),
                nn.ReLU(inplace=True),
            ))
            self.eca.append(ECA(c))
    def forward(self, A, B):
        out = []
        for i, (a, b) in enumerate(zip(A, B)):
            fused = self.proj[i](torch.cat([a, b], dim=1))
            out.append(self.eca[i](fused))
        return out

# --- Concat + CBAM Fusion ---
class FusionConcatCBAM(FusionBase):
    name = "concat_cbam"
    def __init__(self, chs, r=16):
        super().__init__()
        self.proj = nn.ModuleList()
        self.cbam = nn.ModuleList()
        for c in chs:
            self.proj.append(nn.Sequential(
                nn.Conv2d(2 * c, c, 1, bias=False),
                nn.BatchNorm2d(c),
                nn.ReLU(inplace=True),
            ))
            self.cbam.append(CBAM(c, r))
    def forward(self, A, B):
        out = []
        for i, (a, b) in enumerate(zip(A, B)):
            fused = self.proj[i](torch.cat([a, b], dim=1))
            out.append(self.cbam[i](fused))
        return out

class FusionWeightedSum(FusionBase):
    name = "weighted_sum"
    def __init__(self, chs):
        super().__init__()
        self.alpha = nn.Parameter(torch.ones(len(chs)))
        self.beta  = nn.Parameter(torch.ones(len(chs)))
    def forward(self, A, B):
        return [self.alpha[i] * a + self.beta[i] * b
                for i, (a, b) in enumerate(zip(A, B))]

class FusionGated(FusionBase):
    name = "gated"
    def __init__(self, chs, r=16):
        super().__init__()
        self.gates = nn.ModuleList()
        for c in chs:
            mid = max(c // r, 8)
            self.gates.append(nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Conv2d(2 * c, mid, 1), nn.ReLU(inplace=True),
                nn.Conv2d(mid, c, 1), nn.Sigmoid(),
            ))
    def forward(self, A, B):
        return [
            g(torch.cat([a, b], dim=1)) * a + (1 - g(torch.cat([a, b], dim=1))) * b
            for g, a, b in zip(self.gates, A, B)
        ]

class FusionFiLM(FusionBase):
    name = "film"
    def __init__(self, chs, r=16):
        super().__init__()
        self.film = nn.ModuleList()
        for c in chs:
            mid = max(c // r, 8)
            self.film.append(nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Conv2d(c, mid, 1), nn.ReLU(inplace=True),
                nn.Conv2d(mid, 2 * c, 1),
            ))
    def forward(self, A, B):
        out = []
        for i, (a, b) in enumerate(zip(A, B)):
            gb = self.film[i](b)
            gamma, beta = torch.chunk(gb, 2, dim=1)
            out.append((1 + gamma) * a + beta)
        return out

class FusionCrossAttention(FusionBase):
    name = "cross_attn"
    def __init__(self, chs, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.proj_q = nn.ModuleList([nn.Conv2d(c, c, 1) for c in chs])
        self.proj_k = nn.ModuleList([nn.Conv2d(c, c, 1) for c in chs])
        self.proj_v = nn.ModuleList([nn.Conv2d(c, c, 1) for c in chs])
        self.attn   = nn.ModuleList([nn.MultiheadAttention(c, num_heads=num_heads, batch_first=True) for c in chs])
        self.out    = nn.ModuleList([nn.Conv2d(c, c, 1) for c in chs])
    def forward(self, A, B):
        outs = []
        for i, (a, b) in enumerate(zip(A, B)):
            Bn, C, H, W = a.shape
            q = self.proj_q[i](a).flatten(2).transpose(1, 2)
            k = self.proj_k[i](b).flatten(2).transpose(1, 2)
            v = self.proj_v[i](b).flatten(2).transpose(1, 2)
            y, _ = self.attn[i](q, k, v, need_weights=False)
            y = y.transpose(1, 2).reshape(Bn, C, H, W)
            y = self.out[i](y)
            outs.append(a + y)
        return outs

# Updated fusion list with new attention-based fusions
FUSION_LIST = ["late_logits", "concat1x1", "concat_se", "concat_eca", "concat_cbam",
               "weighted_sum", "gated", "film", "cross_attn"]

def build_fusion(name, chs):
    name = name.lower()
    if name == "late_logits":  return FusionLateLogits()
    if name == "concat1x1":    return FusionConcat1x1(chs)
    if name == "concat_se":    return FusionConcatSE(chs)
    if name == "concat_eca":   return FusionConcatECA(chs)
    if name == "concat_cbam":  return FusionConcatCBAM(chs)
    if name == "weighted_sum": return FusionWeightedSum(chs)
    if name == "gated":        return FusionGated(chs)
    if name == "film":         return FusionFiLM(chs)
    if name == "cross_attn":   return FusionCrossAttention(chs, num_heads=4)
    raise ValueError(f"Unknown fusion: {name}")

# ============================================================
# Dual-Swin V2 Model — with Deep Supervision & Channel Attention
# ============================================================
class DualSwinFusionSeg(nn.Module):
    def __init__(self, encoder_name, pretrained, img_size, fpn_channels,
                 fusion_name, decoder_name="hybrid_segformer_unetpp",
                 input_attn="none", post_encoder_attn="none",
                 intra_encoder_attn="none", decoder_output_attn="none"):
        """
        Args:
            encoder_name: Swin V2 model name from timm
            pretrained: Use pretrained weights
            img_size: Input image size
            fpn_channels: Decoder feature channels
            fusion_name: Fusion strategy name
            decoder_name: Decoder architecture name
            input_attn: Input channel attention type ('none', 'se', 'eca', 'cbam')
            post_encoder_attn: Post-encoder attention type ('none', 'se', 'eca', 'cbam')
            intra_encoder_attn: Intra-encoder attention type ('none', 'se', 'eca', 'cbam')
                                Applied INSIDE encoder, after each Swin stage
            decoder_output_attn: Decoder output attention type ('none', 'se', 'eca', 'cbam')
        """
        super().__init__()
        
        # Use intra-attention wrapper if specified
        self.enc_rgb = make_swin_with_intra_attention(
            encoder_name, pretrained=pretrained, img_size=img_size,
            intra_attn=intra_encoder_attn
        )
        self.enc_aux = make_swin_with_intra_attention(
            encoder_name, pretrained=pretrained, img_size=img_size,
            intra_attn=intra_encoder_attn
        )
        
        # Adapt aux encoder for 4 channels
        if intra_encoder_attn.lower() == 'none':
            adapt_patch_embed_in_chans(self.enc_aux, 4)
        else:
            adapt_patch_embed_in_chans(self.enc_aux.base_encoder, 4)

        self.chs = self.enc_rgb.feature_info.channels()
        
        # Attention modules (configurable)
        self.rgb_input_attn = InputChannelAttention(3, attn_type=input_attn)
        self.aux_input_attn = InputChannelAttention(4, attn_type=input_attn)
        self.rgb_post_attn = PostEncoderAttention(self.chs, attn_type=post_encoder_attn)
        self.aux_post_attn = PostEncoderAttention(self.chs, attn_type=post_encoder_attn)
        self.dec_output_attn = DecoderOutputAttention(fpn_channels, attn_type=decoder_output_attn)
        
        self.fusion  = build_fusion(fusion_name, self.chs)
        self.decoder = build_decoder(decoder_name, self.chs, fpn_channels=fpn_channels)
        self.head    = nn.Conv2d(fpn_channels, 1, kernel_size=1)

        self.img_size     = img_size
        self.fusion_name  = fusion_name
        self.decoder_name = decoder_name
        self.is_hybrid    = decoder_name.lower() == "hybrid_segformer_unetpp"
        self.input_attn_type = input_attn
        self.post_encoder_attn_type = post_encoder_attn
        self.intra_encoder_attn_type = intra_encoder_attn
        self.decoder_output_attn_type = decoder_output_attn

    def _encode_rgb(self, rgb):
        rgb = self.rgb_input_attn(rgb)  # Input attention
        feats = self.enc_rgb(rgb)
        feats = to_nchw(feats, self.chs)
        feats = self.rgb_post_attn(feats)  # Post-encoder attention
        return feats

    def _encode_aux(self, aux4):
        aux4 = self.aux_input_attn(aux4)  # Input attention
        feats = self.enc_aux(aux4)
        feats = to_nchw(feats, self.chs)
        feats = self.aux_post_attn(feats)  # Post-encoder attention
        return feats

    def _decode_to_logits(self, feats, return_aux=False):
        if self.is_hybrid:
            x, aux_list = self.decoder(feats)
        else:
            x = self.decoder(feats)
            aux_list = []
        x = self.dec_output_attn(x)  # Decoder output attention
        logits = self.head(x)
        logits = F.interpolate(logits, size=(self.img_size, self.img_size),
                               mode="bilinear", align_corners=False)
        if return_aux:
            # Resize auxiliary logits to full size
            aux_logits = []
            for a in aux_list:
                a_up = F.interpolate(a, size=(self.img_size, self.img_size),
                                     mode="bilinear", align_corners=False)
                aux_logits.append(a_up)
            return {"logits": logits, "aux_logits": aux_logits}
        return logits

    def forward(self, rgb, aux4):
        feats_rgb = self._encode_rgb(rgb)
        feats_aux = self._encode_aux(aux4)

        if isinstance(self.fusion, FusionLateLogits):
            # Late fusion doesn't support deep supervision
            log_rgb = self._decode_to_logits(feats_rgb, return_aux=False)
            log_aux = self._decode_to_logits(feats_aux, return_aux=False)
            return 0.5 * (log_rgb + log_aux)

        feats = self.fusion(feats_rgb, feats_aux)

        if self.training and self.is_hybrid:
            return self._decode_to_logits(feats, return_aux=True)
        else:
            return self._decode_to_logits(feats, return_aux=False)

# ============================================================
# Training / Validation loops — with gradient clipping
# ============================================================
@torch.no_grad()
def validate_loss(model, loader, loss_fn, use_amp=True):
    model.eval()
    total = 0.0; n = 0
    for rgb, aux, mask in tqdm(loader, desc="ValLoss", leave=False):
        rgb  = rgb.to(DEVICE, non_blocking=True)
        aux  = aux.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=(use_amp and DEVICE == "cuda")):
            out = model(rgb, aux)
            logits = out["logits"] if isinstance(out, dict) else out
            loss = loss_fn(logits, mask)
        bs = rgb.size(0)
        total += loss.item() * bs; n += bs
    return total / max(n, 1)

# ============================================================
# TTA (Test-Time Augmentation) — flip ensemble
# ============================================================
@torch.no_grad()
def tta_predict(model, rgb, aux):
    """
    4-fold flip TTA: original + H-flip + V-flip + HV-flip.
    Returns averaged sigmoid probabilities.
    """
    model.eval()
    probs_sum = None
    transforms = [
        lambda r, a: (r, a),                                    # original
        lambda r, a: (r.flip(-1), a.flip(-1)),                  # H-flip
        lambda r, a: (r.flip(-2), a.flip(-2)),                  # V-flip
        lambda r, a: (r.flip(-1).flip(-2), a.flip(-1).flip(-2)),# HV-flip
    ]
    inv_transforms = [
        lambda p: p,
        lambda p: p.flip(-1),
        lambda p: p.flip(-2),
        lambda p: p.flip(-1).flip(-2),
    ]

    for tfm, inv in zip(transforms, inv_transforms):
        r, a = tfm(rgb, aux)
        out = model(r, a)
        logits = out["logits"] if isinstance(out, dict) else out
        probs = inv(torch.sigmoid(logits))
        probs_sum = probs if probs_sum is None else probs_sum + probs

    return probs_sum / len(transforms)

# ============================================================
# Submission writing — .tif output
# ============================================================
def _write_mask_tiff(mask01, out_path, height=128, width=128):
    if mask01.shape[0] != height or mask01.shape[1] != width:
        mask01 = cv2.resize(
            mask01.astype(np.uint8), (width, height),
            interpolation=cv2.INTER_NEAREST,
        )
    with rasterio.open(
        out_path, "w", driver="GTiff",
        height=height, width=width,
        count=1, dtype=rasterio.uint8,
    ) as dst:
        dst.write(mask01.astype(np.uint8), 1)

@torch.no_grad()
def ensemble_predict_tiffs_tta(models, loader, out_dir, thresh=0.5,
                                orig_size=128, use_tta=True):
    """
    K-model ensemble + optional flip TTA.
    Average probabilities → threshold → write .tif at orig_size.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    for m in models:
        m.eval()
    for rgb, aux, src_paths in tqdm(loader, desc="Ensemble+TTA Infer", leave=False):
        rgb = rgb.to(DEVICE, non_blocking=True)
        aux = aux.to(DEVICE, non_blocking=True)
        avg_probs = None
        for m in models:
            if use_tta:
                probs = tta_predict(m, rgb, aux)
            else:
                out = m(rgb, aux)
                logits = out["logits"] if isinstance(out, dict) else out
                probs = torch.sigmoid(logits)
            avg_probs = probs if avg_probs is None else avg_probs + probs
        avg_probs = (avg_probs / len(models)).cpu().numpy()
        for i in range(avg_probs.shape[0]):
            mask01 = (avg_probs[i, 0] > thresh).astype(np.uint8)
            tif_name = Path(src_paths[i]).stem
            out_path = out_dir / f"{tif_name}.tif"
            _write_mask_tiff(mask01, str(out_path), height=orig_size, width=orig_size)

def zip_submission(pred_dir, zip_path):
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for tif_path in sorted(pred_dir.glob("*.tif")):
            zf.write(tif_path, arcname=tif_path.name)

# ============================================================
# K-Fold CV Runner — with deep supervision & gradient clipping
# ============================================================
def run_kfold_experiment(encoder_name, decoder_name, fusion_name, cfg,
                         all_img_paths, all_mask_paths, test_img_paths,
                         scaling_stats, means, stds, pos_weight):
    out_dir  = Path(cfg["out_dir"])
    
    # Build tag including attention config
    attn_tag = ""
    if cfg.get("input_attn", "none") != "none":
        attn_tag += f"_inA{cfg['input_attn']}"
    if cfg.get("post_encoder_attn", "none") != "none":
        attn_tag += f"_peA{cfg['post_encoder_attn']}"
    if cfg.get("intra_encoder_attn", "none") != "none":
        attn_tag += f"_ieA{cfg['intra_encoder_attn']}"
    if cfg.get("decoder_output_attn", "none") != "none":
        attn_tag += f"_doA{cfg['decoder_output_attn']}"
    
    tag      = f"{encoder_name.split('_')[0]}_{decoder_name}_{fusion_name}{attn_tag}"
    ckpt_dir = out_dir / "checkpoints" / tag
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    n_folds = cfg["n_folds"]
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=cfg["seed"])

    geo_aug, rgb_photo_aug, val_aug = build_transforms(
        cfg["img_size"], strong=cfg.get("strong_aug", True))

    fold_results = []
    fold_models = []

    warmup_epochs = cfg.get("warmup_epochs", 3)
    grad_clip     = cfg.get("grad_clip", 1.0)
    aux_weight    = cfg.get("aux_weight", 0.3)
    use_deep_sup  = decoder_name.lower() == "hybrid_segformer_unetpp"

    for fold_idx, (train_indices, val_indices) in enumerate(kf.split(all_img_paths)):
        fold_num = fold_idx + 1
        print(f"\n--- Fold {fold_num}/{n_folds} ---")
        print(f"    Train: {len(train_indices)} | Val: {len(val_indices)}")

        fold_train_imgs  = [all_img_paths[i] for i in train_indices]
        fold_train_masks = [all_mask_paths[i] for i in train_indices]
        fold_val_imgs    = [all_img_paths[i] for i in val_indices]
        fold_val_masks   = [all_mask_paths[i] for i in val_indices]

        train_ds = MarsDualModalSegDataset(
            fold_train_imgs, fold_train_masks,
            RGB_INDICES, AUX_INDICES, scaling_stats, means, stds,
            geo_aug=geo_aug, rgb_photo_aug=rgb_photo_aug, val_aug=None, is_train=True)
        val_ds = MarsDualModalSegDataset(
            fold_val_imgs, fold_val_masks,
            RGB_INDICES, AUX_INDICES, scaling_stats, means, stds,
            geo_aug=None, rgb_photo_aug=None, val_aug=val_aug, is_train=False)

        train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True,
                                  num_workers=cfg["num_workers"], pin_memory=True)
        val_loader   = DataLoader(val_ds, batch_size=cfg["batch_size"], shuffle=False,
                                  num_workers=cfg["num_workers"], pin_memory=True)

        steps_per_epoch = len(train_loader)
        warmup_steps = warmup_epochs * steps_per_epoch

        model = DualSwinFusionSeg(
            encoder_name=encoder_name,
            pretrained=cfg["pretrained"],
            img_size=cfg["img_size"],
            fpn_channels=cfg["fpn_channels"],
            fusion_name=fusion_name,
            decoder_name=decoder_name,
            input_attn=cfg.get("input_attn", "none"),
            post_encoder_attn=cfg.get("post_encoder_attn", "none"),
            intra_encoder_attn=cfg.get("intra_encoder_attn", "none"),
            decoder_output_attn=cfg.get("decoder_output_attn", "none"),
        ).to(DEVICE)

        base_loss_fn = HybridBCEDiceBoundaryLoss(pos_weight=pos_weight)
        if use_deep_sup:
            loss_fn = DeepSupervisionLoss(base_loss_fn, aux_weight=aux_weight)
        else:
            loss_fn = base_loss_fn

        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"],
                                      weight_decay=cfg["weight_decay"])

        _warmup_iters = warmup_epochs * steps_per_epoch
        _total_iters  = cfg["epochs"] * steps_per_epoch
        def lr_lambda(step, _wi=_warmup_iters, _ti=_total_iters):
            if step < _wi:
                return max(step / max(_wi, 1), 0.01)
            progress = (step - _wi) / max(_ti - _wi, 1)
            return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

        scaler = torch.amp.GradScaler("cuda", enabled=(cfg["amp"] and DEVICE == "cuda"))
        ema    = EMA(model, decay=cfg["ema_decay"], warmup_steps=warmup_steps)

        best_miou  = -1.0
        best_epoch = -1
        best_ckpt  = str(ckpt_dir / f"fold{fold_num}_best.pt")
        epoch_logs = []

        for epoch in range(1, cfg["epochs"] + 1):
            model.train()
            total_train_loss = 0.0; n_train = 0
            for rgb, aux, mask in tqdm(train_loader, desc="Train", leave=False):
                rgb  = rgb.to(DEVICE, non_blocking=True)
                aux  = aux.to(DEVICE, non_blocking=True)
                mask = mask.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", enabled=(cfg["amp"] and DEVICE == "cuda")):
                    out = model(rgb, aux)
                    loss = loss_fn(out, mask)
                if scaler is not None and (cfg["amp"] and DEVICE == "cuda"):
                    scaler.scale(loss).backward()
                    if grad_clip > 0:
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    if grad_clip > 0:
                        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    optimizer.step()
                scheduler.step()
                ema.update(model)
                bs = rgb.size(0)
                total_train_loss += loss.item() * bs; n_train += bs
            train_loss = total_train_loss / max(n_train, 1)

            # Validation
            model.eval()
            raw_val_loss = validate_loss(model, val_loader, base_loss_fn, use_amp=cfg["amp"])
            raw_metrics  = compute_leaderboard_metrics_from_loader(
                model, val_loader, thresh=cfg["thresh"])

            ema.apply_shadow(model)
            ema_val_loss = validate_loss(model, val_loader, base_loss_fn, use_amp=cfg["amp"])
            ema_metrics  = compute_leaderboard_metrics_from_loader(
                model, val_loader, thresh=cfg["thresh"])
            ema.restore(model)

            if ema_metrics["mIoU"] >= raw_metrics["mIoU"]:
                val_loss = ema_val_loss; metrics = ema_metrics; use_ema = True
            else:
                val_loss = raw_val_loss; metrics = raw_metrics; use_ema = False

            lr_now = float(optimizer.param_groups[0]["lr"])
            log_entry = {
                "epoch": epoch, "train_loss": float(train_loss),
                "val_loss": float(val_loss), "lr": lr_now, "used_ema": use_ema,
                "raw_mIoU": float(raw_metrics["mIoU"]),
                "ema_mIoU": float(ema_metrics["mIoU"]),
                **{k: float(metrics[k]) for k in
                   ["mIoU", "IoU_fg", "IoU_bg", "F1_fg", "Precision_fg", "Recall_fg"]},
            }
            epoch_logs.append(log_entry)

            ema_tag = "EMA" if use_ema else "RAW"
            print(f"    Ep {epoch:3d}/{cfg['epochs']} | "
                  f"trn={train_loss:.4f} val={val_loss:.4f} | "
                  f"mIoU={metrics['mIoU']:.4f} F1={metrics['F1_fg']:.4f} "
                  f"IoU_fg={metrics['IoU_fg']:.4f} | "
                  f"[{ema_tag}] raw={raw_metrics['mIoU']:.4f} ema={ema_metrics['mIoU']:.4f}")

            if metrics["mIoU"] > best_miou:
                best_miou  = metrics["mIoU"]
                best_epoch = epoch
                if use_ema:
                    ema.apply_shadow(model)
                torch.save({
                    "model": model.state_dict(),
                    "fold": fold_num,
                    "best_epoch": best_epoch,
                    "best_metrics": metrics,
                    "used_ema": use_ema,
                }, best_ckpt)
                if use_ema:
                    ema.restore(model)

        ckpt = torch.load(best_ckpt, map_location=DEVICE)
        model.load_state_dict(ckpt["model"])
        model.eval()
        fold_models.append(model)

        fold_results.append({
            "fold": fold_num,
            "best_epoch": best_epoch,
            "best_ckpt": best_ckpt,
            "best_metrics": ckpt["best_metrics"],
            "epoch_logs": epoch_logs,
            "num_train": len(train_indices),
            "num_val": len(val_indices),
        })
        print(f"    => Fold {fold_num} best mIoU={best_miou:.4f} @ epoch {best_epoch}")

    # Aggregate
    metric_keys = ["mIoU", "IoU_fg", "IoU_bg", "F1_fg", "Precision_fg", "Recall_fg"]
    agg = {}
    for k in metric_keys:
        vals = [fr["best_metrics"][k] for fr in fold_results]
        agg[k] = {"mean": float(np.mean(vals)), "std": float(np.std(vals)),
                   "min": float(np.min(vals)), "max": float(np.max(vals)),
                   "per_fold": vals}

    # Test-set inference with TTA
    test_ds = MarsDualModalSegDataset(
        test_img_paths, None,
        RGB_INDICES, AUX_INDICES, scaling_stats, means, stds,
        geo_aug=None, rgb_photo_aug=None, val_aug=val_aug, is_train=False)
    test_loader = DataLoader(test_ds, batch_size=cfg["batch_size"], shuffle=False,
                             num_workers=cfg["num_workers"], pin_memory=True)

    sub_dir = out_dir / "submissions" / tag
    use_tta = cfg.get("use_tta", True)
    ensemble_predict_tiffs_tta(fold_models, test_loader, sub_dir,
                               thresh=cfg["thresh"], orig_size=cfg["orig_size"],
                               use_tta=use_tta)
    zip_path = str(out_dir / "submissions" / f"{tag}_kfold_ensemble.zip")
    zip_submission(sub_dir, zip_path)

    return {
        "encoder": encoder_name, "decoder": decoder_name, "fusion": fusion_name,
        "input_attn": cfg.get("input_attn", "none"),
        "post_encoder_attn": cfg.get("post_encoder_attn", "none"),
        "intra_encoder_attn": cfg.get("intra_encoder_attn", "none"),
        "decoder_output_attn": cfg.get("decoder_output_attn", "none"),
        "n_folds": n_folds,
        "fold_results": fold_results,
        "aggregate_metrics": agg,
        "ensemble_submission_zip": zip_path,
        "num_params": sum(p.numel() for p in fold_models[0].parameters()),
    }

print("All definitions loaded (v6 — Hybrid SegFormer×UNet++ with Deep Supervision, "
      "Boundary+Lovász loss, TTA, Channel Attention).")
print(f"Available Decoders: {DECODER_LIST}")
print(f"Available Fusions:  {FUSION_LIST}")
print(f"Available Attention Types: {ATTENTION_TYPES}")

In [ ]:

# ============================================================
# EXPERIMENT CONFIGURATION (v6-DA — Hybrid Decoder, ARCHIVE data, Per-Image Norm)
# ============================================================

cfg = dict(
    # --- Data ---
    data_root   = "../../data/phase1_dataset",
    out_dir     = "../../kfold_results_v6_p1-86_DA",

    # --- Architecture ---
    encoders    = ["swinv2_small_window8_256"],
    decoders    = ["hybrid_segformer_unetpp"],   # ★ The new hybrid decoder
    fusions     = ["concat_eca"],                 # "concat_se", "concat_eca", "concat_cbam", etc.

    # --- Channel Attention Strategies ---
    # Options for each: "none", "se", "eca", "cbam"
    # 
    # Location A: Input attention (before encoder) — learns which raw channels matter
    input_attn          = "eca",   # "none", "se", "eca", "cbam"
    
    # Location B: Post-encoder attention (after all Swin stages) — refines encoder features
    post_encoder_attn   = "none",   # "none", "se", "eca", "cbam"
    
    # Location E: Intra-encoder attention (INSIDE encoder, after EACH Swin stage) — deep refinement
    intra_encoder_attn  = "se",   # "none", "se", "eca", "cbam"
    
    # Location D: Decoder output attention (before head) — refines final features
    decoder_output_attn = "cbam",   # "none", "se", "eca", "cbam"

    # --- K-Fold ---
    n_folds     = 5,

    # --- Training ---
    img_size    = 128,        # ★ native resolution — no upscaling
    orig_size   = 128,
    epochs      = 60,
    batch_size  = 16,         # may need 12 if OOM (hybrid is slightly heavier)
    num_workers = 2,
    lr          = 2e-4,
    weight_decay= 1e-4,
    seed        = 42,
    pretrained  = True,
    fpn_channels= 256,
    amp         = True,
    ema_decay   = 0.995,
    warmup_epochs = 3,
    thresh      = 0.5,

    # --- v6 additions ---
    strong_aug  = True,       # stronger augmentations (elastic, grid, coarse dropout)
    grad_clip   = 1.0,        # gradient clipping max norm
    aux_weight  = 0.3,        # deep supervision auxiliary loss weight
    use_tta     = True,        # test-time augmentation (flip ensemble)
    max_stat_files = None,     # cap files used for stats computation (None = all)
)

# ============================================================
# ATTENTION STRATEGY GUIDE
# ============================================================
# 
# FUSION-LEVEL ATTENTION (in fusions list):
#   - concat_se:   SE (Squeeze-Excitation) — standard, ~4K params per level
#   - concat_eca:  ECA (Efficient Channel Attention) — only ~5 params per level, fast!
#   - concat_cbam: CBAM (Channel + Spatial) — most expressive, ~5K params per level
#
# ADDITIONAL ATTENTION LOCATIONS:
#   - input_attn:          Weight raw input channels (RGB: 3ch, AUX: 4ch)
#   - post_encoder_attn:   Refine encoder features after all stages
#   - intra_encoder_attn:  Inject attention INSIDE encoder, after EACH Swin stage (deep refinement)
#   - decoder_output_attn: Refine final decoder output
#
# RECOMMENDED COMBOS:
#   1. Baseline:    fusion="concat_se", all others="none"
#   2. Lightweight: fusion="concat_eca", decoder_output_attn="eca"
#   3. Maximum:     fusion="concat_cbam", input_attn="eca", intra_encoder_attn="se", decoder_output_attn="cbam"
#   4. Deep Intra:  fusion="concat_se", intra_encoder_attn="eca"
#
# ============================================================

total_runs = len(cfg["encoders"]) * len(cfg["decoders"]) * len(cfg["fusions"])
print(f"Total experiment configs: {total_runs}")
print(f"K-Folds per config: {cfg['n_folds']}")
print(f"Total training runs: {total_runs * cfg['n_folds']}")
print(f"\nDecoder: {cfg['decoders']} ★ HYBRID SegFormer×UNet++")
print(f"Encoders: {cfg['encoders']}")
print(f"Fusions:  {cfg['fusions']}")
print(f"\nAttention Strategy:")
print(f"  Input attention:          {cfg['input_attn']}")
print(f"  Post-encoder attention:   {cfg['post_encoder_attn']}")
print(f"  Intra-encoder attention:  {cfg['intra_encoder_attn']}")
print(f"  Decoder output attention: {cfg['decoder_output_attn']}")
print(f"\nStrong augmentations: {cfg['strong_aug']}")
print(f"Gradient clipping:    {cfg['grad_clip']}")
print(f"Deep supervision aux: {cfg['aux_weight']}")
print(f"Test-Time Augment:    {cfg['use_tta']}")
print(f"Loss: BCE(0.3) + Dice(0.3) + Boundary(0.2) + Lovász(0.2)")

steps_per_epoch_est = 868 // cfg["batch_size"] + 1
warmup_steps_est = cfg["warmup_epochs"] * steps_per_epoch_est
print(f"\nEMA decay: {cfg['ema_decay']} | Warmup: {cfg['warmup_epochs']} epochs (~{warmup_steps_est} steps)")


In [ ]:

# ============================================================
# RUN K-FOLD CROSS-VALIDATION (v6-DA — Domain-Adapted, Per-Image Normalization)
# ============================================================
set_seed(cfg["seed"])
print("DEVICE:", DEVICE)
print("CFG:", json.dumps({k: v for k, v in cfg.items()
                          if not isinstance(v, (list, np.ndarray))}, indent=2))

data_root = Path(cfg["data_root"])

train_img_dir  = data_root / "train" / "images"
train_mask_dir = data_root / "train" / "masks"
val_img_dir    = data_root / "val" / "images"
val_mask_dir   = data_root / "val" / "masks"
test_img_dir   = data_root / "test" / "images"

train_imgs = sorted(list(train_img_dir.glob("*.tif")))
val_imgs   = sorted(list(val_img_dir.glob("*.tif")))
all_img_paths = train_imgs + val_imgs

all_mask_paths = []
for img_p in all_img_paths:
    if str(img_p).startswith(str(train_img_dir)):
        mask_p = train_mask_dir / img_p.name
    else:
        mask_p = val_mask_dir / img_p.name
    assert mask_p.exists(), f"Mask not found: {mask_p}"
    all_mask_paths.append(mask_p)

test_img_paths = sorted(list(test_img_dir.glob("*.tif")))

print(f"Total labeled samples (train+val): {len(all_img_paths)}")
print(f"  - From train/: {len(train_imgs)}")
print(f"  - From val/:   {len(val_imgs)}")
print(f"Test images: {len(test_img_paths)}")

# ★ v6-DA: Per-image normalization — no global percentile stats needed
# compute_scaling_stats is NOT used; we pass a dummy stats dict for interface compat
BAND_INDICES_ALL = list(range(1, 8))  # rasterio is 1-based; TIFs have bands 1-7
scaling_stats = {"low": np.zeros(len(BAND_INDICES_ALL)),
                 "high": np.ones(len(BAND_INDICES_ALL)),
                 "p_low": 1.0, "p_high": 99.0}  # placeholder — not used in Dataset
means, stds = compute_mean_std_per_image_norm(
    all_img_paths, BAND_INDICES_ALL,
    p_low=1.0, p_high=99.0, max_files=cfg.get("max_stat_files"))
print(f"  ★ v6-DA: Using per-image percentile normalization (domain-invariant)")
print(f"  ★ Channel means after per-image norm: {means}")
print(f"  ★ Channel stds  after per-image norm: {stds}")
fg_frac, pos_weight = compute_pos_weight(all_mask_paths)
print(f"FG frac: {fg_frac:.6f} | pos_weight: {pos_weight:.2f}")

out_dir = Path(cfg["out_dir"])
out_dir.mkdir(parents=True, exist_ok=True)

# ★ v6-DA: Save normalization stats to JSON for inference
norm_stats_path = out_dir / "norm_stats_v6_DA.json"
norm_stats_payload = {
    "normalization": "per_image_percentile",
    "p_low": 1.0,
    "p_high": 99.0,
    "band_indices_all": BAND_INDICES_ALL,
    "rgb_indices": RGB_INDICES,
    "aux_indices": AUX_INDICES,
    "channel_means": means.tolist(),
    "channel_stds": stds.tolist(),
    "pos_weight": pos_weight,
    "fg_frac": fg_frac,
    "img_size": cfg["img_size"],
}
norm_stats_path.write_text(json.dumps(norm_stats_payload, indent=2))
print(f"  ★ Normalization stats saved to: {norm_stats_path}")

all_results = []
total = len(cfg["encoders"]) * len(cfg["decoders"]) * len(cfg["fusions"])
run_idx = 0

for enc_name in cfg["encoders"]:
    for dec_name in cfg["decoders"]:
        for fus_name in cfg["fusions"]:
            run_idx += 1
            print(f"\n{'=' * 80}")
            print(f"CONFIG {run_idx}/{total}: encoder={enc_name}  "
                  f"decoder={dec_name}  fusion={fus_name}")
            print(f"Running {cfg['n_folds']}-Fold Cross-Validation...")
            print("=" * 80)
            result = run_kfold_experiment(
                enc_name, dec_name, fus_name, cfg,
                all_img_paths, all_mask_paths, test_img_paths,
                scaling_stats, means, stds, pos_weight)
            all_results.append(result)

all_results = sorted(all_results,
                     key=lambda r: r["aggregate_metrics"]["mIoU"]["mean"], reverse=True)

report_json = out_dir / "kfold_report_v6_DA.json"
payload = {
    "experiment_id": EXP["id"],
    "channels": EXP["channels"],
    "rgb_indices": RGB_INDICES,
    "aux_indices": AUX_INDICES,
    "pos_weight": pos_weight,
    "total_labeled_samples": len(all_img_paths),
    "n_folds": cfg["n_folds"],
    "cfg": {k: v for k, v in cfg.items() if not isinstance(v, np.ndarray)},
    "results": all_results,
}
report_json.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
print(f"\nReport saved: {report_json}")

print("\n" + "=" * 80)
print("K-FOLD CROSS-VALIDATION SUMMARY (v6-DA — Hybrid SegFormer×UNet++, Per-Image Norm)")
print("=" * 80)
for r in all_results:
    agg = r["aggregate_metrics"]
    print(f"\n{r['encoder']} / {r['decoder']} / {r['fusion']}")
    print(f"  mIoU:      {agg['mIoU']['mean']:.4f} ± {agg['mIoU']['std']:.4f}")
    print(f"  IoU_fg:    {agg['IoU_fg']['mean']:.4f} ± {agg['IoU_fg']['std']:.4f}")
    print(f"  IoU_bg:    {agg['IoU_bg']['mean']:.4f} ± {agg['IoU_bg']['std']:.4f}")
    print(f"  F1_fg:     {agg['F1_fg']['mean']:.4f} ± {agg['F1_fg']['std']:.4f}")
    print(f"  Precision: {agg['Precision_fg']['mean']:.4f} ± {agg['Precision_fg']['std']:.4f}")
    print(f"  Recall:    {agg['Recall_fg']['mean']:.4f} ± {agg['Recall_fg']['std']:.4f}")
    print(f"  Per-fold mIoU: {agg['mIoU']['per_fold']}")
    print(f"  Ensemble submission: {r['ensemble_submission_zip']}")
    print(f"  Params: {r['num_params']:,}")

print("\nDONE.")


In [ ]:
# ============================================================
# Visualize K-Fold Results (v6)
# ============================================================
import matplotlib.pyplot as plt

for r in all_results:
    agg = r["aggregate_metrics"]
    tag = f"{r['encoder'].split('_')[0]} / {r['decoder']} / {r['fusion']}"

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    fold_mious = agg["mIoU"]["per_fold"]
    folds_x = list(range(1, len(fold_mious) + 1))

    ax = axes[0]
    bars = ax.bar(folds_x, fold_mious, color="steelblue", edgecolor="black", alpha=0.8)
    ax.axhline(y=agg["mIoU"]["mean"], color="red", linestyle="--", linewidth=2,
               label=f"Mean = {agg['mIoU']['mean']:.4f}")
    ax.fill_between([0.5, len(folds_x) + 0.5],
                    agg["mIoU"]["mean"] - agg["mIoU"]["std"],
                    agg["mIoU"]["mean"] + agg["mIoU"]["std"],
                    alpha=0.15, color="red", label=f"±1 std = {agg['mIoU']['std']:.4f}")
    ax.set_xlabel("Fold"); ax.set_ylabel("mIoU")
    ax.set_title(f"{tag}\nPer-Fold mIoU")
    ax.set_xticks(folds_x); ax.legend()
    ax.set_ylim(0, max(fold_mious) * 1.15)

    ax = axes[1]
    for fr in r["fold_results"]:
        epochs = [e["epoch"] for e in fr["epoch_logs"]]
        mious  = [e["mIoU"]  for e in fr["epoch_logs"]]
        ax.plot(epochs, mious, label=f"Fold {fr['fold']}", alpha=0.7)
    ax.set_xlabel("Epoch"); ax.set_ylabel("mIoU")
    ax.set_title(f"{tag}\nValidation mIoU per Epoch")
    ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(cfg["out_dir"]) / f"kfold_plot_{r['decoder']}_{r['fusion']}.png", dpi=150)
    plt.show()

print("\n" + "=" * 100)
print(f"{'Encoder':<30s} | {'Decoder':<28s} | {'Fusion':<12s} | {'mIoU':>12s} | {'F1_fg':>12s} | {'IoU_fg':>12s}")
print("-" * 100)
for r in all_results:
    agg = r["aggregate_metrics"]
    print(f"{r['encoder']:<30s} | {r['decoder']:<28s} | {r['fusion']:<12s} | "
          f"{agg['mIoU']['mean']:.4f}±{agg['mIoU']['std']:.4f} | "
          f"{agg['F1_fg']['mean']:.4f}±{agg['F1_fg']['std']:.4f} | "
          f"{agg['IoU_fg']['mean']:.4f}±{agg['IoU_fg']['std']:.4f}")
print("=" * 100)